# Geometry of Truth

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JeffVallyath/geometry-of-truth/blob/v1.0.0/notebooks/geometry_of_truth.ipynb)

## Why this experiment exists

The larger study asks whether a language model represents the relation between a situation and a consideration, such as whether privacy supports one action but opposes another. Before interpreting that relation experiment, the activation pipeline needs to recover a known semantic distinction when surface answer symbols change. This control comes first.

A probe could appear successful by tracking the token used for an answer. The experiment breaks that shortcut by reversing the meanings of A and B during training and testing, then transferring the fitted procedure to unseen 1 and 2 answers. A stable truth direction under both changes shows that the pipeline reads factual class information from the hidden state rather than one preferred answer symbol.

The confirmatory result selects layer 14, finds a signed separation of 1.924455 training standard deviations, measures directional consensus of 0.998695 on a zero to one scale, obtains a permutation p value of 0.000999, and retains a separation of 1.840165 after the answer symbols change to 1 and 2.

## Start here

From GitHub, click the Open in Colab badge above. In Colab, choose Runtime and Run all. Demo mode verifies the public result files and renders the analysis immediately. Full reconstruction downloads the model and factual datasets, extracts the activations again, and requires an accepted model license, an HF_TOKEN Colab secret, and a CUDA GPU with at least 23,000 MiB of free memory. The memory threshold is an execution requirement rather than an experimental result.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PUBLIC_REPOSITORY = 'https://github.com/JeffVallyath/geometry-of-truth.git'
PUBLIC_REF = 'v1.0.0'
RUN_MODE = 'DEMO'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path("/content/geometry-of-truth")
    if (REPO_ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", "tag", PUBLIC_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", PUBLIC_REF], check=True)
    elif not (REPO_ROOT / "pyproject.toml").is_file():
        if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
            raise RuntimeError("The Colab repository directory exists but is not a usable checkout")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", PUBLIC_REF, PUBLIC_REPOSITORY, str(REPO_ROOT)],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)], check=True)
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(path for path in candidates if (path / "pyproject.toml").is_file())

SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

print({"mode": RUN_MODE})

In [ ]:
from IPython.display import display

from geometry_of_truth.truth.contracts import load_bundle, number_lineage
from geometry_of_truth.truth.plots import layer_selection, permutation_null
from geometry_of_truth.truth.results import (
    consensus,
    design_counts,
    direction_method,
    mapping_checks,
    partition_effects,
    permutation_summary,
    prompt_examples,
    transfer,
    v1_diagnostics,
)

bundle = load_bundle(REPO_ROOT)
v1 = bundle["v1"]
v2 = bundle["v2"]
print("Artifact integrity verified")

## Experimental setup

The model is Meta Llama 3.1 8B Instruct. For each prompt, the extractor records every transformer layer at the final question token before the model produces its answer. The probe therefore receives an internal state from the same point in the response process for every example.

The factual sources contribute 1,496 affirmative city records and 1,496 matched negated records to each answer scheme. The experiment forms 748 proposition groups and keeps every linked version in one split, preventing an affirmative form from training a probe that later sees its negated partner in testing. Groups stay intact.

Each answer scheme contains 2,992 prompt records. Within each scheme, 1,796 records train the directions, 600 select the layer, and 596 remain untouched for the confirmatory test. The held-out 1 and 2 scheme mirrors the A and B design, producing 5,984 cached records across both schemes.

Standard prompts map true to A and false to B. Reversed prompts map true to B and false to A. The transfer prompts replace those symbols with 1 and 2. Every semantic class therefore appears under both answer assignments.

In [ ]:
display(prompt_examples())
display(design_counts(bundle["split"]))
display(mapping_checks(bundle["split"], v2))

## Frozen first attempt

The first design used the literal words True and False while instructing the model to reverse their meanings. AUROC measures ranking quality from zero to one, with 0.5 representing chance. Under the reversed instruction, semantic scoring reached 0.000349 while literal True minus False scoring reached 0.999651. The model followed the familiar words almost perfectly and ignored their assigned meanings almost perfectly.

That version remains a formal failure. The neutral-symbol design was specified afterward and keeps the failed result visible instead of rewriting the original rule.

In [ ]:
print(v1["frozen_v1_disposition"])
v1_table = v1_diagnostics(v1)
decisive_v1_rows = [
    "reversed instruction mapped macro AUROC",
    "reversed literal True minus False macro AUROC",
]
display(v1_table[v1_table["quantity"].isin(decisive_v1_rows)].reset_index(drop=True))

## Activation measure

The code divides the training records into eight partitions. For each held-out partition and each layer, it subtracts the mean hidden state for false training rows from the mean for true training rows, then normalizes that vector to length one. The held-out partition never contributes to its own direction.

Each test activation is projected onto its training direction, centered with training class means, and divided by the training projection standard deviation. One partition effect is the mean true score minus the mean false score, with the two factual sources weighted equally. The signed statistic T averages the eight partition effects. T has no fixed upper bound, and a positive value means the training orientation transfers to held-out data.

Directional consensus C is the length of the average of the eight unit directions. It ranges from zero to one. A value near one means that separate training partitions recover nearly the same direction.

In [ ]:
display(direction_method())
display(consensus(v2))

## Layer selection

The development sweep evaluates layers 8 through 24, with the horizontal axis showing the transformer layer and the vertical axis showing signed development separation for each answer mapping. A candidate layer must separate truth correctly under both standard and reversed A and B mappings. The selection score uses the weaker of those two effects, so one easy mapping cannot determine the winner. Layer 14 maximizes that rule. Test data remains untouched until this choice is fixed.

In [ ]:
display(layer_selection(v2).figure)

## Confirmatory result

At layer 14, all eight held-out partition effects are positive and range from 1.876134 to 1.955230. Their average is T equal to 1.924455, meaning that true and false test projections differ by about 1.92 training standard deviations after equal weighting of the two factual sources.

The eight unit directions produce C equal to 0.998695 and an implied mean pairwise cosine of 0.997019. Both quantities show that the direction changes very little across training partitions.

The null keeps groups intact. The permutation test flips labels for complete proposition groups, preserving every linked affirmative and negated set. Zero of 1,000 shuffled results reach T equal to 1.924455. The prespecified add-one calculation gives 1 divided by 1,001, or p equal to 0.000999. This p value measures incompatibility with the group-preserving null, while T measures the size of the separation.

In [ ]:
primary_effects = partition_effects(v2)
primary_effects = primary_effects[
    (primary_effects["verbalizer"] == "A/B") & (primary_effects["mapping"] == "overall")
][["partition", "signed effect"]]
display(primary_effects.reset_index(drop=True))
permutation_table = permutation_summary(v2)
shown_fields = ["permutations", "observed T", "null values at least observed", "add one p"]
display(permutation_table[permutation_table["field"].isin(shown_fields)].reset_index(drop=True))
display(permutation_null(v2).figure)

## Symbol transfer

The selected layer and fitted procedure next score prompts that use 1 and 2 instead of A and B. The table reports standard mapping, reversed mapping, and their equal-weight average for both symbol schemes. Signed T retains the training orientation. Macro AUROC reports ranking quality, where 0.5 is chance and 1.0 is perfect ranking within each factual source.

For A and B, standard mapping gives T equal to 2.033345 and reversed mapping gives 1.815513, producing the 1.924455 average. For held-out 1 and 2, the corresponding values are 1.987243 and 1.694375, producing the 1.840165 average. Macro AUROC equals 1.0 in every row. The decrease preserves strong separation while removing the original answer tokens.

In [ ]:
display(transfer(v2))

## Where every headline number comes from

Every displayed result comes from the public result bundle loaded at the start of the notebook. Integrity verification runs before any result table. The final two columns below name the data file and exact field or calculation for each headline quantity, allowing a reader to trace the prose back to stored values.

In [ ]:
display(number_lineage(bundle))

## Reproduction modes

DEMO verifies and presents the retained result files. ANALYSIS recomputes every statistic from a retained activation cache after checking every cache part. FULL downloads the fixed inputs, extracts activations, runs the analysis, and compares the reproduced measurements with the public reference.

Analysis mode reads the cache location from TRUTH_CACHE_ROOT. Full mode writes generated files inside the active Colab runtime.

In [ ]:
if RUN_MODE == "ANALYSIS":
    from geometry_of_truth.truth.reproduce import reproduce_analysis

    cache_root = os.environ.get("TRUTH_CACHE_ROOT")
    if not cache_root:
        raise RuntimeError("Set TRUTH_CACHE_ROOT to the retained activation cache")
    analysis_run = reproduce_analysis(cache_root, "/content/truth-analysis")
    display(analysis_run["comparison"])
else:
    print("Analysis reconstruction skipped")

In [ ]:
if RUN_MODE == "FULL":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[truth-full]"],
        check=True,
    )
    if IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("Add HF_TOKEN to Colab secrets and enable notebook access")
    from geometry_of_truth.truth.reproduce import reproduce_full; full_run = reproduce_full("/content/truth-reproduction"); display(full_run["checks"])
else:
    print("Set RUN_MODE to FULL and rerun from the first cell for raw reconstruction")

## Interpretation

The control establishes that this extraction and cross-fit procedure can recover a known semantic distinction after two answer-mapping changes. That result makes a later positive endorsement finding more credible and makes a null endorsement result harder to attribute to broken activation extraction, while the next stage asks whether ValuePrism can support the same measurement under unseen consideration identities and reciprocal checkerboards.